
# 📘 KNN Student Pack (v3) — With Data Overview & Custom Predictions
Datasets: **Iris**, **Wine**, **Breast Cancer**

**Workflow:** Overview → Split/Scale → Train → Evaluate → Tune *k* → Predict → Reflect


## 0) Setup & Imports

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Setup complete")


## 1) Helper Functions

In [ ]:

def dataset_overview(data, feature_names, target, target_names=None, target_col="target"):
    df = pd.DataFrame(data, columns=feature_names)
    df[target_col] = target
    if target_names is not None:
        df[target_col + "_name"] = df[target_col].map(dict(enumerate(target_names)))
    print("Shape:", df.shape)
    display(df.head())
    display(df.describe().round(2))
    return df

def split_scale(X, y, test_size=0.2, random_state=42, stratify=True):
    strat = y if stratify else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=strat
    )
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test), y_train, y_test, scaler

def fit_knn(X_train, y_train, k=5):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    return model

def evaluate(model, X_test, y_test, target_names=None):
    y_pred = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

def sweep_k(X_train, X_test, y_train, y_test, ks=range(1, 26), title="KNN Accuracy vs k"):
    scores = []
    for k in ks:
        m = KNeighborsClassifier(n_neighbors=k)
        m.fit(X_train, y_train)
        scores.append(m.score(X_test, y_test))
    plt.figure()
    plt.plot(list(ks), scores, marker='o')
    plt.title(title)
    plt.xlabel("k (neighbours)")
    plt.ylabel("Accuracy")
    plt.grid(True)
    plt.show()
    return scores


## 2) Iris 🌸

In [ ]:

iris = load_iris()
df_iris = dataset_overview(iris.data, iris.feature_names, iris.target, iris.target_names, target_col="species")

X_i = df_iris.drop(columns=["species","species_name"]).values
y_i = df_iris["species"].values

Xi_tr, Xi_te, yi_tr, yi_te, scaler_i = split_scale(X_i, y_i, test_size=0.2)
model_i = fit_knn(Xi_tr, yi_tr, k=3)
evaluate(model_i, Xi_te, yi_te, target_names=iris.target_names)
_ = sweep_k(Xi_tr, Xi_te, yi_tr, yi_te, title="Iris — Accuracy vs k")

new_sample = np.array([[5.9, 3.0, 5.1, 1.8]])
pred_i = model_i.predict(scaler_i.transform(new_sample))[0]
print("Predicted species:", iris.target_names[pred_i])

plt.figure(figsize=(6,5))
plt.scatter(Xi_tr[:,2], Xi_tr[:,3], c=yi_tr, alpha=0.6)
ns = scaler_i.transform(new_sample)
plt.scatter(ns[0,2], ns[0,3], marker="X", s=150)
plt.xlabel("Petal length (scaled)")
plt.ylabel("Petal width (scaled)")
plt.title(f"Custom Sample → {iris.target_names[pred_i]}")
plt.grid(True)
plt.show()


## 3) Wine 🍷

In [ ]:

wine = load_wine()
df_wine = dataset_overview(wine.data, wine.feature_names, wine.target, wine.target_names, target_col="class")

X_w = df_wine.drop(columns=["class","class_name"]).values
y_w = df_wine["class"].values

Xw_tr, Xw_te, yw_tr, yw_te, scaler_w = split_scale(X_w, y_w, test_size=0.2)
model_w = fit_knn(Xw_tr, yw_tr, k=5)
evaluate(model_w, Xw_te, yw_te, target_names=wine.target_names)
_ = sweep_k(Xw_tr, Xw_te, yw_tr, yw_te, title="Wine — Accuracy vs k")

new_sample_w = np.array([[13.0, 2.0, 2.4, 16.0, 100.0,
                          2.8, 3.1, 0.3, 2.0, 5.0,
                          1.0, 3.0, 1000.0]])
pred_w = model_w.predict(scaler_w.transform(new_sample_w))[0]
print("Predicted wine class:", wine.target_names[pred_w])


## 4) Breast Cancer 🎗️

In [ ]:

bc = load_breast_cancer()
df_bc = dataset_overview(bc.data, bc.feature_names, bc.target, bc.target_names, target_col="class")

X_b = df_bc.drop(columns=["class","class_name"]).values
y_b = df_bc["class"].values

Xb_tr, Xb_te, yb_tr, yb_te, scaler_b = split_scale(X_b, y_b, test_size=0.3)
model_b = fit_knn(Xb_tr, yb_tr, k=3)
evaluate(model_b, Xb_te, yb_te, target_names=bc.target_names)
_ = sweep_k(Xb_tr, Xb_te, yb_tr, yb_te, title="Breast Cancer — Accuracy vs k")

new_sample_b = np.array([[17.0, 10.0, 120.0, 1000.0, 0.10,
                          0.15, 0.20, 0.10, 0.20, 0.06,
                          0.50, 1.0, 3.0, 40.0, 0.006,
                          0.02, 0.02, 0.01, 0.02, 0.003,
                          20.0, 15.0, 140.0, 1200.0, 0.14,
                          0.30, 0.40, 0.20, 0.30, 0.08]])
pred_b = model_b.predict(scaler_b.transform(new_sample_b))[0]
print("Predicted class:", bc.target_names[pred_b])


## 5) Reflection ✍️


- Record your best **k** for each dataset.  
- Explain in 2–3 sentences why **scaling** matters for KNN.  
- For Breast Cancer, which matters more — **precision** or **recall** — and why?  
